In [9]:

import websocket
import json
import csv
import datetime
import os
import pandas as pd
import threading
import time
from sqlalchemy import create_engine
import urllib.parse

In [2]:

WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
CSV_FILE_NAME = "data.csv"

In [3]:

def tao_file_csv():
    header = ["thoi_gian", "gia", "gia_mua", "gia_ban", "khoi_luong_24h","high24h", "low24h","instId","best_purchase_price", "best_sale_price"]
    
    if not os.path.exists(CSV_FILE_NAME) or os.path.getsize(CSV_FILE_NAME) == 0:
        with open(CSV_FILE_NAME, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV: {CSV_FILE_NAME}")

tao_file_csv()

In [ ]:

def on_open(ws):
    print(f" Đã kết nối thành công")
    
    
    for inst_id in INSTRUMENT_IDS:
        subscribe_message = {
            "op": "subscribe",
            "args": [
                {    
                    "instType": "USDT-FUTURES",
                    "channel": "ticker",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(subscribe_message))
        print(f"Đang theo dõi {inst_id}")
    print(f"Đang theo dõi các cặp: {', '.join(INSTRUMENT_IDS)}")

all_data= []

def on_message(ws, message_str):
    global all_data
    data = json.loads(message_str)
    all_data.append(data)
    
    if "data" in data and data["data"]:
        ticker = data["data"][0]
        
        
        thoi_gian = datetime.datetime.now().isoformat()
        gia = ticker.get('lastPr')
        gia_mua = ticker.get('bidPr')
        gia_ban = ticker.get('askPr')
        khoi_luong = ticker.get('volumeUsd24h')
        high_int_24h= ticker.get('high24h')
        low_int_24h = ticker.get('low24h')
        instId = ticker.get('instId') 
        best_purchase_price= ticker.get('bidSz')
        best_sale_price = ticker.get('askSz')
        
        
        # Hiển thị
        print(f" {instId} | Giá: {gia}") 
        
        # Lưu vào CSV
        with open(CSV_FILE_NAME, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([thoi_gian, gia, gia_mua, gia_ban, khoi_luong, high_int_24h, low_int_24h, instId, best_purchase_price, best_sale_price])

def on_error(ws, error):
    print(f" Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f" Kết nối đã đóng")

In [5]:
a=60  #1 phút
b=a*60  #1h
c=b*24  #1 ngày
def run_ws():
    ws.run_forever(ping_interval=30, ping_timeout=10)
ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open,
                          on_message=on_message,
                          on_error=on_error,
                          on_close=on_close)

print(f"Bắt đầu kết nối đến Bitget...")
print(f"Dữ liệu sẽ được lưu vào: {CSV_FILE_NAME}")
print("Nhấn Ctrl+C để dừng")

ws_thread = threading.Thread(target=run_ws)
ws_thread.daemon = True
ws_thread.start()

run_duration = 10

try:
    time.sleep(run_duration)
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")

# Đóng kết nối sau thời gian quy định
ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")

# nhìn time ở trên tự tính hoặc để mặc định 
# add time ở phần Run_duration
# rồi chạy cell dataframe
# muốn xóa  phần add vào csv thì xóa phần CSV_FILE_NAME = "data.csv" ở trên


Bắt đầu kết nối đến Bitget...
Dữ liệu sẽ được lưu vào: data.csv
Nhấn Ctrl+C để dừng
 Đã kết nối thành công
Đang theo dõi SOLUSDT
Đang theo dõi BTCUSDT
Đang theo dõi ETHUSDT
Đang theo dõi các cặp: SOLUSDT, BTCUSDT, ETHUSDT
 ETHUSDT | Giá: 2693.11
 BTCUSDT | Giá: 108521.4
 SOLUSDT | Giá: 171.926
 ETHUSDT | Giá: 2693.11
 BTCUSDT | Giá: 108521.4
 SOLUSDT | Giá: 171.926
 ETHUSDT | Giá: 2693.11
 BTCUSDT | Giá: 108521.4
 SOLUSDT | Giá: 171.926
 BTCUSDT | Giá: 108521.4
 ETHUSDT | Giá: 2693.11
 SOLUSDT | Giá: 171.926
 ETHUSDT | Giá: 2693.11
 BTCUSDT | Giá: 108521.4
 SOLUSDT | Giá: 171.926
 BTCUSDT | Giá: 108521.5
 ETHUSDT | Giá: 2693.12
 SOLUSDT | Giá: 171.923
 ETHUSDT | Giá: 2693.12
 BTCUSDT | Giá: 108521.5
 SOLUSDT | Giá: 171.923
 BTCUSDT | Giá: 108521.5
 ETHUSDT | Giá: 2693.11
 SOLUSDT | Giá: 171.923
 ETHUSDT | Giá: 2693.11
 BTCUSDT | Giá: 108521.5
 SOLUSDT | Giá: 171.923
 BTCUSDT | Giá: 108521.5
 SOLUSDT | Giá: 171.923
 ETHUSDT | Giá: 2693.11
 BTCUSDT | Giá: 108521.5
 ETHUSDT | Giá: 2693.11

In [6]:

ws.close()


data_for_df = []
for msg in all_data:
    if "data" in msg and msg["data"]:
        ticker = msg["data"][0]
        
        row = {
            "thoi_gian": datetime.datetime.now().isoformat(), 
            "gia": ticker.get('lastPr'),
            "gia_mua": ticker.get('bidPr'),
            "gia_ban": ticker.get('askPr'),
            "khoi_luong_24h": ticker.get('volumeUsd24h'),
            "high24h": ticker.get('high24h'),
            "low24h": ticker.get('low24h'),
            "instId": ticker.get('instId'),
            "best_purchase_price": ticker.get('bidSz'),
            "best_sale_price": ticker.get('askSz')
        }
        data_for_df.append(row)


df = pd.DataFrame(data_for_df)



In [ ]:

host = 'localhost'  
port = '5432'
database = 'postgres'  
username = 'postgres'
password = 'postgres'  


conn_url = f'postgresql://{username}:{urllib.parse.quote_plus(password)}@{host}:{port}/{database}'
engine = create_engine(conn_url)

try:
    
    df['thoi_gian'] = pd.to_datetime(df['thoi_gian'])
    
    
    df = df.rename(columns={'instId': 'instid'})
    
    # Lưu vào database
    df.to_sql('coin_prices', engine, if_exists='append', index=False, 
              method='multi', chunksize=1000)
    print(f"Đã lưu thành công {len(df)} dòng dữ liệu vào TimescaleDB")
except Exception as e:
    print(f"Lỗi khi lưu dữ liệu: {e}")
#bảng    
# CREATE TABLE coin_prices (
#     id SERIAL PRIMARY KEY,
#     thoi_gian TIMESTAMPTZ NOT NULL,
#     gia NUMERIC(20, 8),
#     gia_mua NUMERIC(20, 8),
#     gia_ban NUMERIC(20, 8),
#     khoi_luong_24h NUMERIC(30, 8),
#     high24h NUMERIC(20, 8),
#     low24h NUMERIC(20, 8),
#     instId TEXT,  
#     best_purchase_price NUMERIC(20, 8),
#     best_sale_price NUMERIC(20, 8)
# );

Đã lưu thành công 66 dòng dữ liệu vào TimescaleDB


In [8]:
query = "SELECT * FROM coin_prices LIMIT 100"
pd.read_sql(query, engine)

,id,thoi_gian,gia,gia_mua,gia_ban,khoi_luong_24h,high24h,low24h,instid,best_purchase_price,best_sale_price
0,1,2025-05-29 05:47:15.719149+00:00,107241.500,107241.500,107241.600,None,109234.50,106745.900,BTCUSDT,26.0563,8.4041
1,2,2025-05-29 05:47:15.719149+00:00,2641.210,2641.200,2641.210,None,2688.63,2608.000,ETHUSDT,101.0000,64.1500
2,3,2025-05-29 05:47:15.719149+00:00,170.434,170.434,170.435,None,177.38,168.797,SOLUSDT,0.6000,1080.1000
3,4,2025-05-29 05:47:15.719149+00:00,2641.210,2641.200,2641.210,None,2688.63,2608.000,ETHUSDT,141.3000,2.0200
4,5,2025-05-29 05:47:15.719149+00:00,107241.500,107241.500,107241.600,None,109234.50,106745.900,BTCUSDT,26.0563,8.4041
...,...,...,...,...,...,...,...,...,...,...,...
95,96,2025-05-29 05:56:23.347752+00:00,170.866,170.866,170.867,None,177.38,168.797,SOLUSDT,212.6000,132.6000
96,97,2025-05-29 05:56:23.347752+00:00,2648.520,2648.520,2648.530,None,2688.63,2608.000,ETHUSDT,107.6100,99.3000
97,98,2025-05-29 05:56:23.347752+00:00,107389.800,107389.700,107389.800,None,109234.50,106745.900,BTCUSDT,26.6994,1.1030
98,99,2025-05-29 05:56:23.347752+00:00,170.866,170.866,170.867,None,177.38,168.797,SOLUSDT,1090.0000,132.6000


-- Xem toàn bộ dữ liệu (có giới hạn)
SELECT * FROM coin_prices LIMIT 100;
